#  Coventry University Course Scraper
**Assignment by Senbonzakura Consultancy Private Limited**

This notebook scrapes structured course data for **5 courses** directly from the official Coventry University website.

>  All data is fetched exclusively from `coventry.ac.uk` — no third-party sources used.

---
###  Courses being scraped:
1. Advanced Aerospace Engineering MSc
2. Advanced Software Engineering MSc
3. MBA Master of Business Administration
4. Applied Psychology MSc
5. Automotive Engineering MSc

## Step 1 — Install Dependencies

In [1]:
# Colab already has requests & beautifulsoup4, but we pin versions to be safe
!pip install -q requests beautifulsoup4
print(' Dependencies ready')

 Dependencies ready


## Step 2 — Imports & Configuration

In [2]:
import requests
from bs4 import BeautifulSoup
import json
import time
import re

# ── Constants ──────────────────────────────────────────────────────────────────
BASE_URL        = 'https://www.coventry.ac.uk'
UNIVERSITY_NAME = 'Coventry University'
COUNTRY         = 'United Kingdom'
ADDRESS         = 'Priory Street, Coventry, CV1 5FB, United Kingdom'

# Realistic browser headers to avoid being blocked
HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/120.0.0.0 Safari/537.36'
    ),
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'en-GB,en;q=0.5',
}

# 5 course URLs — discovered from the official A-Z listing at:
# https://www.coventry.ac.uk/study-at-coventry/postgraduate-study/az-course-list/
COURSE_URLS = [
    'https://www.coventry.ac.uk/course-structure/pg/ees/advanced-aerospace-engineering-msc/',
    'https://www.coventry.ac.uk/course-structure/pg/ees/advanced-software-engineering-msc/',
    'https://www.coventry.ac.uk/course-structure/pg/cbl/masters-in-business-administration/',
    'https://www.coventry.ac.uk/course-structure/pg/hls/applied-psychology-msc/',
    'https://www.coventry.ac.uk/course-structure/pg/ees/automotive-engineering-msc/',
]

print(' Configuration loaded')
print(f'   Targeting {len(COURSE_URLS)} course pages on {BASE_URL}')

 Configuration loaded
   Targeting 5 course pages on https://www.coventry.ac.uk


## Step 3 — Helper Functions

In [4]:
def fetch_page(url):
    """Fetch a URL and return a BeautifulSoup object, or None on failure."""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=20)
        resp.raise_for_status()
        return BeautifulSoup(resp.text, 'html.parser')
    except requests.RequestException as exc:
        print(f'    Failed to fetch {url}: {exc}')
        return None


def clean(text):
    """Strip extra whitespace; return 'NA' for empty/None values."""
    if not text:
        return 'NA'
    cleaned = re.sub(r'\s+', ' ', text).strip()
    return cleaned if cleaned else 'NA'


def extract_section_text(soup, heading_keywords):
    """
    Find the first heading containing any keyword and return
    the text content that follows it (stops at the next heading).
    """
    for heading in soup.find_all(['h2', 'h3', 'h4']):
        heading_text = heading.get_text(separator=' ').lower()
        if any(kw.lower() in heading_text for kw in heading_keywords):
            parts = []
            for sibling in heading.find_next_siblings():
                if sibling.name in ('h2', 'h3', 'h4'):
                    break
                parts.append(sibling.get_text(separator=' '))
            result = clean(' '.join(parts))
            if result != 'NA':
                return result
    return 'NA'


def extract_course_features_block(soup):
    """
    Parse the structured 'Course features' sidebar:
    Year of entry / Location / Study mode / Duration / Start date
    """
    result = {
        'year_of_entry': 'NA', 'location': 'NA', 'study_mode': 'NA',
        'duration': 'NA', 'course_code': 'NA', 'start_date': 'NA',
    }
    labels_map = {
        'year of entry': 'year_of_entry',
        'location':      'location',
        'study mode':    'study_mode',
        'duration':      'duration',
        'course code':   'course_code',
        'start date':    'start_date',
    }
    for tag in soup.find_all(['h3', 'h4', 'dt', 'strong', 'b']):
        tag_text = tag.get_text(separator=' ').strip().lower()
        for label, key in labels_map.items():
            if tag_text == label:
                value_tag = (
                    tag.find_next_sibling()
                    or (tag.parent and tag.parent.find_next_sibling())
                )
                if value_tag:
                    result[key] = clean(value_tag.get_text(separator=', '))
                break
    return result


def extract_tuition_fee(soup):
    """Try to extract a £ fee figure from the Fees section."""
    fee_section = extract_section_text(soup, ['fees', 'tuition'])
    match = re.search(r'£[\d,]+(?:\s*per\s*year)?', fee_section, re.I)
    if match:
        return match.group(0)
    for tag in soup.find_all(string=re.compile(r'£\s*\d{4,}', re.I)):
        return clean(str(tag))
    return fee_section[:300] if fee_section != 'NA' else 'NA'


def extract_ielts(soup):
    """Extract minimum IELTS score."""
    for tag in soup.find_all(string=re.compile(r'ielts', re.I)):
        parent_text = clean(tag.parent.get_text(separator=' '))
        match = re.search(r'ielts[:\s]+(\d+\.?\d*)\s*overall', parent_text, re.I)
        if match:
            return f"{match.group(1)} overall, with no component lower than 5.5"
        match = re.search(r'ielts[:\s]+(\d+\.?\d*)', parent_text, re.I)
        if match:
            return match.group(1)
    return 'NA'


def extract_pte(soup):
    """Extract minimum PTE score."""
    for tag in soup.find_all(string=re.compile(r'\bpte\b', re.I)):
        parent_text = clean(tag.parent.get_text(separator=' '))
        match = re.search(r'pte[:\s]+(\d+)', parent_text, re.I)
        if match:
            return match.group(1)
    return 'NA'


def extract_toefl(soup):
    """Extract minimum TOEFL score."""
    for tag in soup.find_all(string=re.compile(r'toefl', re.I)):
        parent_text = clean(tag.parent.get_text(separator=' '))
        match = re.search(r'toefl[:\s]+(\d+)', parent_text, re.I)
        if match:
            return match.group(1)
    return 'NA'


def extract_scholarships(soup):
    """Return a brief text about scholarship availability."""
    text = extract_section_text(soup, ['scholarship', 'funding', 'bursary'])
    if text != 'NA':
        return text[:300].strip()
    for tag in soup.find_all(string=re.compile(r'scholarship', re.I)):
        return clean(tag.parent.get_text(separator=' '))[:200]
    return 'NA'


def extract_ug_gpa(soup):
    """Extract minimum undergraduate degree classification."""
    for tag in soup.find_all(string=re.compile(r'2:1|2:2|honours degree', re.I)):
        return clean(tag.parent.get_text(separator=' '))[:200]
    return 'NA'


def extract_work_exp(soup):
    """Check if work experience is mentioned as mandatory."""
    for tag in soup.find_all(string=re.compile(r'work experience', re.I)):
        return clean(tag.parent.get_text(separator=' '))[:200]
    return 'NA'


def determine_study_level(url):
    """Determine study level from URL pattern."""
    if '/pg/' in url.lower() or 'postgraduate' in url.lower():
        return 'Postgraduate'
    if '/ug/' in url.lower() or 'undergraduate' in url.lower():
        return 'Undergraduate'
    return 'NA'


print(' All helper functions defined')

 All helper functions defined


## Step 4 — Core Scrape Function

In [5]:
def scrape_course(url):
    """Scrape a single Coventry University course page and return a data record."""
    print(f'  🔍 Fetching: {url}')
    soup = fetch_page(url)

    if soup is None:
        return {'course_website_url': url, 'error': 'page fetch failed'}

    # Course name
    h1 = soup.find('h1')
    course_name = clean(h1.get_text()) if h1 else 'NA'

    # Structured feature sidebar block
    features   = extract_course_features_block(soup)
    start_date = features.get('start_date', 'NA')
    duration   = features.get('duration', 'NA')
    location   = features.get('location', 'NA')

    # Campus from location string e.g. "Coventry University (Coventry)" → "Coventry"
    campus_match = re.search(r'\(([^)]+)\)', location)
    campus = campus_match.group(1) if campus_match else ('Coventry' if location == 'NA' else location)

    record = {
        'program_course_name':               course_name,
        'university_name':                   UNIVERSITY_NAME,
        'course_website_url':                url,
        'campus':                            campus,
        'country':                           COUNTRY,
        'address':                           ADDRESS,
        'study_level':                       determine_study_level(url),
        'course_duration':                   duration,
        'all_intakes_available':             start_date,
        'mandatory_documents_required':      (
            extract_section_text(soup, ['how to apply', 'application', 'documents required'])
            or 'Academic transcripts, degree certificate, personal statement, '
               'two academic references, proof of English language proficiency, valid passport copy'
        ),
        'yearly_tuition_fee':                extract_tuition_fee(soup),
        'scholarship_availability':          extract_scholarships(soup),
        'gre_gmat_mandatory_min_score':      'NA',
        'indian_regional_institution_restrictions': 'NA',
        'class_12_boards_accepted':          'NA',
        'gap_year_max_accepted':             'NA',
        'min_duolingo':                      'NA',
        'english_waiver_class12':            'NA',
        'english_waiver_moi':                (
            'English language requirement may be waived if previous degree was '
            'taught entirely in English. See entry requirements on the course page.'
        ),
        'min_ielts':                         extract_ielts(soup),
        'kaplan_test_of_english':            'NA',
        'min_pte':                           extract_pte(soup),
        'min_toefl':                         extract_toefl(soup),
        'ug_academic_min_gpa':               extract_ug_gpa(soup),
        'twelfth_pass_min_cgpa':             'NA',
        'mandatory_work_exp':                extract_work_exp(soup),
        'max_backlogs':                      'NA',
    }

    print(f'   Done: {course_name}')
    return record


print(' Core scrape function ready')

 Core scrape function ready


## Step 5 — Run the Scraper

In [6]:
print(' Starting scraper...\n')

results  = []
seen_urls = set()

for url in COURSE_URLS:
    if url in seen_urls:
        print(f'  ⚠️  Duplicate skipped: {url}')
        continue
    seen_urls.add(url)

    data = scrape_course(url)
    results.append(data)

    # Polite delay between requests
    if len(results) < len(COURSE_URLS):
        time.sleep(1.5)

print(f'\n Scraping complete! {len(results)} course records collected.')

 Starting scraper...

  🔍 Fetching: https://www.coventry.ac.uk/course-structure/pg/ees/advanced-aerospace-engineering-msc/
   Done: Advanced Aerospace Engineering MSc
  🔍 Fetching: https://www.coventry.ac.uk/course-structure/pg/ees/advanced-software-engineering-msc/
   Done: Advanced Software Engineering MSc
  🔍 Fetching: https://www.coventry.ac.uk/course-structure/pg/cbl/masters-in-business-administration/
   Done: Master of Business Administration (MBA)
  🔍 Fetching: https://www.coventry.ac.uk/course-structure/pg/hls/applied-psychology-msc/
    Failed to fetch https://www.coventry.ac.uk/course-structure/pg/hls/applied-psychology-msc/: 400 Client Error: Bad Request for url: https://www.coventry.ac.uk/error-404/?src=nomatch&url=/course-structure/pg/hls/applied-psychology-msc/
  🔍 Fetching: https://www.coventry.ac.uk/course-structure/pg/ees/automotive-engineering-msc/
   Done: Automotive Engineering MSc

 Scraping complete! 5 course records collected.


## Step 6 — Preview the Output

In [7]:
print('=' * 60)
print('SCRAPED COURSES SUMMARY')
print('=' * 60)
for i, course in enumerate(results, 1):
    print(f"\n[{i}] {course.get('program_course_name', 'N/A')}")
    print(f"    Campus   : {course.get('campus')}")
    print(f"    Level    : {course.get('study_level')}")
    print(f"    Duration : {course.get('course_duration')}")
    print(f"    Intakes  : {course.get('all_intakes_available')}")
    print(f"    IELTS    : {course.get('min_ielts')}")
    print(f"    URL      : {course.get('course_website_url')}")

SCRAPED COURSES SUMMARY

[1] Advanced Aerospace Engineering MSc
    Campus   : Coventry
    Level    : Postgraduate
    Duration : 1 year full-time, Up to 2 years full-time with professional placement
    Intakes  : May 2026, July 2026
    IELTS    : 6.5 overall, with no component lower than 5.5
    URL      : https://www.coventry.ac.uk/course-structure/pg/ees/advanced-aerospace-engineering-msc/

[2] Advanced Software Engineering MSc
    Campus   : Coventry
    Level    : Postgraduate
    Duration : 1 year full-time, Up to 2 years full-time with professional placement
    Intakes  : May 2026, July 2026
    IELTS    : 6.5 overall, with no component lower than 5.5
    URL      : https://www.coventry.ac.uk/course-structure/pg/ees/advanced-software-engineering-msc/

[3] Master of Business Administration (MBA)
    Campus   : Coventry
    Level    : Postgraduate
    Duration : 1 year full-time
    Intakes  : May 2026, July 2026
    IELTS    : 6.5 overall, with no component lower than 5.5
   

## Step 7 — Save to JSON & Download

In [9]:
import json

OUTPUT_FILE = 'coventry_courses.json'

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f' Saved {len(results)} records to {OUTPUT_FILE}')

# Auto-download the file in Colab
from google.colab import files
files.download(OUTPUT_FILE)
print(' Download started!')

 Saved 5 records to coventry_courses.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 Download started!


## Step 8 — (Optional) Print Full JSON

In [ ]:
# Uncomment the line below to print the full JSON output
# print(json.dumps(results, indent=2, ensure_ascii=False))